In [1]:
import os
import calendar
from datetime import date
from pathlib import Path
import pandas as pd
import numpy as np
import clickhouse_connect
from dotenv import load_dotenv

### Configuration & Setup

In [2]:
load_dotenv()  # Load secrets from a .env file securely

YEAR = 2026
MONTH = 7  # Change this to process different months

BASE_DIR = Path("/Volumes/E$/CEIR/Clean Dumps")  # Update this to your actual base directory
FAKE_DUMP_PATH    = BASE_DIR / "Fake"    / f"fake_{YEAR}-{MONTH:02d}.parquet"
GENUINE_DUMP_PATH = BASE_DIR / "Genuine" / f"genuine_{YEAR}-{MONTH:02d}.parquet"
CLONED_DUMP_PATH  = BASE_DIR / "Cloned"  / f"cloned_{YEAR}-{MONTH:02d}.parquet"

TIME_COL = "imei_first_seen"   # Timestamp column driving the month filter
TOP_N    = 50                  # Rank depth for top-model / top-IMEI tables

# Optional: run OPTIMIZE TABLE ... FINAL after each insert so
# ReplacingMergeTree collapses re-run duplicates immediately
OPTIMIZE_AFTER_INSERT = True

# --- CLICKHOUSE CONNECTION ---
# Make sure your .env file uses these CH_ variables
DB = dict(
    host=os.environ.get("CH_HOST", "localhost"),
    port=int(os.environ.get("CH_PORT", 8123)),
    database=os.environ.get("CH_DATABASE", "default"),
    username=os.environ.get("CH_USER", "default"),
    password=os.environ.get("CH_PASSWORD", ""),
)

### Helper Functions

In [3]:
def month_end_period(year: int, month: int) -> date:
    """Returns the exact last day of the given month."""
    last_day = calendar.monthrange(year, month)[1]
    return date(year, month, last_day)

In [4]:
# FEATURE PREP — derive time parts + age brackets from raw data
# ============================================================
def prep_time_and_age(df: pd.DataFrame, time_col: str = TIME_COL) -> pd.DataFrame:
    # Ensure the timestamp column is real datetime before using .dt
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

    # Time dimensions used across the monthly reports
    df["year_month"] = df[time_col].dt.to_period("M")
    df["hour"]       = df[time_col].dt.hour
    df["dow"]        = df[time_col].dt.day_name()

    # Age -> ordered bracket. right=True means each bin is (low, high];
    # include_lowest=True so an exact age of 0 still lands in "<18".
    if "age" in df.columns:
        bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
        labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100-120"]
        numeric_age = pd.to_numeric(df["age"], errors="coerce")
        age_bracket = pd.cut(numeric_age, bins=bins, labels=labels,
                             right=True, include_lowest=True)

        # Missing / out-of-range ages -> "unknown" (no age data on file)
        age_bracket = age_bracket.astype("object").fillna("unknown")
        df["age_bracket"] = pd.Categorical(age_bracket,
                                           categories=labels + ["unknown"],
                                           ordered=True)

    # Technology -> highest generation the device supports (5G > 4G > 3G > 2G).
    # Only possible on frames carrying the GSMA capability flags (genuine).
    if {"has_2g", "has_3g", "has_4g", "has_5g"}.issubset(df.columns):
        conditions = [
            df["has_5g"] == 1,
            df["has_4g"] == 1,
            df["has_3g"] == 1,
            df["has_2g"] == 1,
        ]
        choices = ["5G", "4G", "3G", "2G"]
        df["technology"] = pd.Categorical(
            np.select(conditions, choices, default="unknown"),
            categories=["2G", "3G", "4G", "5G", "unknown"],
            ordered=True)
    return df

In [5]:
# FEATURE PREP — cloned dataset (pre-aggregated, one row per IMEI)
# ============================================================
def prep_clone(df: pd.DataFrame, time_col: str = TIME_COL) -> pd.DataFrame:
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
    df["year_month"] = df[time_col].dt.to_period("M")
    # Fake = TAC not found in GSMA (all enrichment columns null together)
    df["status"] = df["oem"].isna().map({True: "FAKE", False: "GENUINE"})
    return df

In [6]:
def add_time_dimensions(out: pd.DataFrame) -> pd.DataFrame:
    """Expands the year_month Period column into period / year / month
    and drops the raw Period (ClickHouse cannot ingest pandas Periods)."""
    out["year"]   = out["year_month"].dt.year.astype(int)
    out["month"]  = out["year_month"].dt.month.astype(int)
    out["period"] = [month_end_period(y, m) for y, m in zip(out["year"], out["month"])]
    return out.drop(columns=["year_month"])

### Data Aggregation Builders

In [7]:
def build_by_dimension(
    df: pd.DataFrame,
    status: str,
    dim_col: str,
    *,
    fill: str | None = None,
    dropna: bool = True,
) -> pd.DataFrame:
    """Long-format equivalent of report_by_month(): device counts per
    year_month x dimension, tagged with status ('fake' / 'genuine').
    Frames lacking the dimension (e.g. fake_df has no GSMA columns)
    contribute nothing rather than raising."""
    cols_order = ["period", "year", "month", "status", dim_col, "imei_count"]
    if dim_col not in df.columns:
        return pd.DataFrame(columns=cols_order)

    series = df[dim_col]
    # Fill nulls when a label is given (cast to object first so it works
    # even if the column is categorical, e.g. mno/gender/age_bracket)
    if fill is not None:
        series = series.astype("object").fillna(fill)

    out = (
        df.assign(**{dim_col: series})
          .groupby(["year_month", dim_col], observed=True, dropna=dropna)
          .size()
          .reset_index(name="imei_count")
    )

    out = add_time_dimensions(out)
    out.insert(0, "status", status)
    out[dim_col] = out[dim_col].astype(str)   # Categoricals/NaN -> plain strings

    return out[cols_order]

In [8]:
def build_month_summ(df: pd.DataFrame, status: str) -> pd.DataFrame:
    """High-level monthly snapshot per status — the 'TOTAL' margin of the
    crosstab reports (IMEI counts per month)."""
    out = (
        df.groupby("year_month", observed=True)
          .agg(imei_count=("year_month", "size"))
          .reset_index()
    )

    out = add_time_dimensions(out)
    out.insert(0, "status", status)

    cols_order = ["period", "year", "month", "status", "imei_count"]
    return out[cols_order]

In [9]:
def build_combined(builder, *args, **kwargs) -> pd.DataFrame:
    """Runs a builder for both fake and genuine frames and stacks the result
    so each table carries a status dimension instead of being duplicated."""
    return pd.concat(
        [
            builder(fake_df, "fake", *args, **kwargs),
            builder(genuine_df, "genuine", *args, **kwargs),
        ],
        ignore_index=True,
    )

In [10]:
# TOP-N BRAND/MODEL — long-format equivalent of report_top_brand_model()
# ============================================================
def build_top_model(df: pd.DataFrame, status: str, n: int = TOP_N) -> pd.DataFrame:
    """Top-n brand/model per month. Adapts to the dataset grain:
    record-level frames (genuine) count distinct IMEIs and IMSIs from raw
    columns; the pre-aggregated clone frame sums its imsi_count instead."""
    g = df.dropna(subset=["brand"])

    aggs = {"imei_count": ("imei", "nunique")}
    if "imsi_count" in g.columns:                 # pre-aggregated (clone)
        aggs["imsi_count"] = ("imsi_count", "sum")
    elif "imsi" in g.columns:                     # record-level (genuine)
        aggs["imsi_count"] = ("imsi", "nunique")

    out = (
        g.groupby(["year_month", "brand", "model"], observed=True, dropna=False)
         .agg(**aggs)
         .reset_index()
         .sort_values(["year_month", "imei_count"], ascending=[True, False])
         .groupby("year_month", group_keys=False)
         .head(n)
         .reset_index(drop=True)
    )
    out["rank"] = out.groupby("year_month").cumcount() + 1

    out = add_time_dimensions(out)
    out.insert(0, "status", status)
    for c in ("brand", "model"):
        out[c] = out[c].astype(str)

    cols_order = ["period", "year", "month", "status", "rank",
                  "brand", "model", "imei_count", "imsi_count"]
    return out[[c for c in cols_order if c in out.columns]]

In [11]:
# CLONE BUILDERS — monthly status split + top-N IMEIs
# ============================================================
def build_clone_month_summ(df: pd.DataFrame) -> pd.DataFrame:
    """Cloned IMEIs per month split by FAKE/GENUINE TAC status."""
    out = (
        df.groupby(["year_month", "status"], observed=True)["imei"]
          .nunique()
          .reset_index(name="imei_count")
    )
    out = add_time_dimensions(out)
    cols_order = ["period", "year", "month", "status", "imei_count"]
    return out[cols_order]


def build_clone_top_imei(df: pd.DataFrame, n: int = TOP_N) -> pd.DataFrame:
    """Top-n cloned IMEIs per month, ranked by imsi_count (number of
    distinct subscribers sharing the IMEI — the severity of the cloning)."""
    keep = ["year_month", "imei", "status", "imsi_count", "msisdn_count",
            "brand", "model"]
    out = (
        df.sort_values(["year_month", "imsi_count"], ascending=[True, False])
          .groupby("year_month", group_keys=False)
          .head(n)
          .loc[:, [c for c in keep if c in df.columns]]
          .reset_index(drop=True)
    )
    out["rank"] = out.groupby("year_month").cumcount() + 1

    out = add_time_dimensions(out)
    for c in ("brand", "model"):
        if c in out.columns:
            out[c] = out[c].astype("object").fillna("unknown").astype(str)

    cols_order = ["period", "year", "month", "rank", "imei", "status",
                  "imsi_count", "msisdn_count", "brand", "model"]
    return out[[c for c in cols_order if c in out.columns]]


def build_clone_by_dimension(df: pd.DataFrame, dim_col: str,
                             *, fill: str = "unknown") -> pd.DataFrame:
    """Cloned IMEIs per month per GSMA dimension (device_type / os_family).
    Fake-status clones have no GSMA metadata and land in the fill bucket."""
    series = df[dim_col].astype("object").fillna(fill)
    out = (
        df.assign(**{dim_col: series})
          .groupby(["year_month", dim_col], observed=True)["imei"]
          .nunique()
          .reset_index(name="imei_count")
    )
    out = add_time_dimensions(out)
    out[dim_col] = out[dim_col].astype(str)
    cols_order = ["period", "year", "month", dim_col, "imei_count"]
    return out[cols_order]

### Execution Pipeline

In [12]:
if __name__ == "__main__":

    # 1. Load fake, genuine and cloned IMEI datasets
    print("Loading fake, genuine and cloned Parquet dumps...")
    fake_df    = pd.read_parquet(FAKE_DUMP_PATH)
    genuine_df = pd.read_parquet(GENUINE_DUMP_PATH)
    clone_df   = pd.read_parquet(CLONED_DUMP_PATH)

    # Standardize uppercase column to match database schema
    for frame in (fake_df, genuine_df):
        if "MNO" in frame.columns:
            frame.rename(columns={"MNO": "mno"}, inplace=True)

    # 2. Memory optimization: Convert low-cardinality strings to categories
    print("Optimizing memory with categorical casting...")
    for frame in (fake_df, genuine_df):
        for c in ("mno", "gender", "district", "country"):
            if c in frame.columns:
                frame[c] = frame[c].astype("category")

    # 3. Feature prep: time parts, age brackets, technology, clone status
    print("Deriving time dimensions, age brackets and clone status...")
    fake_df    = prep_time_and_age(fake_df)
    genuine_df = prep_time_and_age(genuine_df)
    clone_df   = prep_clone(clone_df)

Loading fake, genuine and cloned Parquet dumps...
Optimizing memory with categorical casting...
Deriving time dimensions, age brackets and clone status...


In [13]:
    # 4. Define the processing manifest
    PUBLISH = [
        # ---- existing fake/genuine tables --------------------------------
        {
            "database": "ceir",
            "table": "imei_month_summ",
            "build": lambda: build_combined(build_month_summ),
        },
        {
            "database": "ceir",
            "table": "imei_by_mno",
            "build": lambda: build_combined(build_by_dimension, "mno", fill="unknown"),
        },
        {
            "database": "ceir",
            "table": "imei_by_gender",
            "build": lambda: build_combined(build_by_dimension, "gender", fill="unknown"),
        },
        {
            "database": "ceir",
            "table": "imei_by_age",
            "build": lambda: build_combined(build_by_dimension, "age_bracket", dropna=False),
        },
        {
            "database": "ceir",
            "table": "imei_by_district",
            "build": lambda: build_combined(build_by_dimension, "district"),
        },
        {
            "database": "ceir",
            "table": "imei_by_country",
            "build": lambda: build_combined(build_by_dimension, "country", fill="unknown"),
        },
        # ---- new GSMA-dimension tables (genuine only carries GSMA cols;
        #      build_combined still works because fake_df returns empty) ----
        {
            "database": "ceir",
            "table": "imei_by_device_type",
            "build": lambda: build_combined(build_by_dimension, "device_type", fill="unknown"),
        },
        {
            "database": "ceir",
            "table": "imei_by_os_family",
            "build": lambda: build_combined(build_by_dimension, "os_family", fill="unknown"),
        },
        {
            "database": "ceir",
            "table": "imei_by_technology",
            "build": lambda: build_combined(build_by_dimension, "technology", dropna=False),
        },
        # ---- new top-N brand/model table (genuine) -----------------------
        {
            "database": "ceir",
            "table": "imei_top_model",
            "build": lambda: build_top_model(genuine_df, "genuine", n=TOP_N),
        },
        # ---- new cloned-IMEI tables ---------------------------------------
        {
            "database": "ceir",
            "table": "clone_month_summ",
            "build": lambda: build_clone_month_summ(clone_df),
        },
        {
            "database": "ceir",
            "table": "clone_top_imei",
            "build": lambda: build_clone_top_imei(clone_df, n=TOP_N),
        },
        {
            "database": "ceir",
            "table": "clone_top_model",
            "build": lambda: build_top_model(clone_df, "cloned", n=TOP_N),
        },
        {
            "database": "ceir",
            "table": "clone_by_device_type",
            "build": lambda: build_clone_by_dimension(clone_df, "device_type"),
        },
        {
            "database": "ceir",
            "table": "clone_by_os_family",
            "build": lambda: build_clone_by_dimension(clone_df, "os_family"),
        },
    ]

In [14]:
    # 5. Execute database transactions
    print("\nConnecting to ClickHouse database...")
    client = clickhouse_connect.get_client(**DB)

    try:
        for item in PUBLISH:
            print(f"Building data for {item['database']}.{item['table']}...")
            df_out = item["build"]()
            df_out["updated_at"] = pd.Timestamp.now()

            print(f"Inserting into {item['database']}.{item['table']} ({len(df_out):,} rows)...")
            client.insert_df(
                table=item["table"],
                database=item["database"],
                df=df_out,
            )

            # ReplacingMergeTree dedups lazily; force the collapse so
            # re-runs of the same months don't show duplicate rows
            if OPTIMIZE_AFTER_INSERT:
                client.command(
                    f"OPTIMIZE TABLE {item['database']}.{item['table']} FINAL"
                )

        print("\n✅ Pipeline completed successfully. All data committed.")

    except Exception as e:
        print(f"\n❌ Pipeline failed. Error: {e}")
        raise

    finally:
        client.close()
        print("Database connection closed.")


Connecting to ClickHouse database...
Building data for ceir.imei_month_summ...
Inserting into ceir.imei_month_summ (2 rows)...
Building data for ceir.imei_by_mno...
Inserting into ceir.imei_by_mno (8 rows)...
Building data for ceir.imei_by_gender...
Inserting into ceir.imei_by_gender (8 rows)...
Building data for ceir.imei_by_age...
Inserting into ceir.imei_by_age (16 rows)...
Building data for ceir.imei_by_district...
Inserting into ceir.imei_by_district (272 rows)...
Building data for ceir.imei_by_country...
Inserting into ceir.imei_by_country (188 rows)...
Building data for ceir.imei_by_device_type...
Inserting into ceir.imei_by_device_type (15 rows)...
Building data for ceir.imei_by_os_family...
Inserting into ceir.imei_by_os_family (33 rows)...
Building data for ceir.imei_by_technology...
Inserting into ceir.imei_by_technology (5 rows)...
Building data for ceir.imei_top_model...
Inserting into ceir.imei_top_model (50 rows)...
Building data for ceir.clone_month_summ...
Inserting i

### Cleanup

In [15]:
client.close()